In [56]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

products_data = [
        (1, "Хлеб"),
        (2, "Молоко"),
        (3, "Яблоко"),
        (4, "Сок"),
        (5, "Кофе")
    ]
categories_data = [
        (1, "Еда"),
        (2, "Напитки"),
        (3, "Фрукты")
    ]
product_category_data = [
        (1, 1),  # Хлеб - Еда
        (2, 1),  # Молоко - Еда
        (3, 3),  # Яблоко - Фрукты
        (4, 2)   # Сок - Напитки
        # Кофе не имеет категории
    ]

In [57]:
class Warehouse:
  def __init__(self, name) -> None:
    self.__spark = SparkSession.builder.appName(name).master("local[*]").getOrCreate()

    self.__df_product = None
    self.__df_categories = None
    self.__df_product_category = None

  @property
  def product(self):
    return self.__df_product

  @product.setter
  def product(self, data):
    self.__df_product = self.__spark.createDataFrame(data, schema=["id", "name"])
    self.__df_product.createOrReplaceTempView("products")


  @property
  def categories(self):
    return self.__df_categories

  @categories.setter
  def categories(self, data):
    self.__df_categories = self.__spark.createDataFrame(data, schema=["id", "name"])
    self.__df_categories.createOrReplaceTempView("categories")

  @property
  def relation_prod2cat(self):
    return self.__df_product_category

  @relation_prod2cat.setter
  def relation_prod2cat(self, data):
    self.__df_product_category = self.__spark.createDataFrame(data, schema=["product_id", "category_id"])
    self.__df_product_category.createOrReplaceTempView("product_category")


  def productInfo(self):
    return self.__spark.sql("""
      select p.name as `товар` , c.name as `категория`
      from  products p
      left join product_category pc on p.id = pc.product_id
      left join categories c on c.id = pc.category_id
    """)



In [58]:
wh = Warehouse("warehouse1")

wh.product = products_data
wh.categories = categories_data
wh.relation_prod2cat = product_category_data

products_info = wh.productInfo()

products_info.show()

+------+---------+
| товар|категория|
+------+---------+
|  Кофе|     NULL|
|  Хлеб|      Еда|
|Молоко|      Еда|
|Яблоко|   Фрукты|
|   Сок|  Напитки|
+------+---------+



Тесты

In [59]:
product_name = [el[1] for el in products_data]
products_result = [row['товар'] for row in products_info.collect()]

product_name, products_result

(['Хлеб', 'Молоко', 'Яблоко', 'Сок', 'Кофе'],
 ['Кофе', 'Хлеб', 'Молоко', 'Яблоко', 'Сок'])

In [60]:
for p in product_name:
  assert p in products_result

"Проверка пройдена"

'Проверка пройдена'